<a href="https://colab.research.google.com/github/mfyuce/deep-learning-with-python-notebooks/blob/master/chapter09_convnet-architecture-patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [1]:
!pip install keras keras-hub --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.21.1 requires keras-hub==0.21.1, but you have keras-hub 0.26.0 which is incompatible.


In [2]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [3]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## ConvNet architecture patterns

### Modularity, hierarchy, and reuse

### Residual connections

In [4]:
import keras
from keras import layers

inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, activation="relu")(inputs)
residual = x
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
residual = layers.Conv2D(64, 1)(residual)
x = layers.add([x, residual])

In [5]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, activation="relu")(inputs)
residual = x
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D(2, padding="same")(x)
residual = layers.Conv2D(64, 1, strides=2)(residual)
x = layers.add([x, residual])

In [6]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Rescaling(1.0 / 255)(inputs)

def residual_block(x, filters, pooling=False):
    residual = x
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    if pooling:
        x = layers.MaxPooling2D(2, padding="same")(x)
        residual = layers.Conv2D(filters, 1, strides=2)(residual)
    elif filters != residual.shape[-1]:
        residual = layers.Conv2D(filters, 1)(residual)
    x = layers.add([x, residual])
    return x

x = residual_block(x, filters=32, pooling=True)
x = residual_block(x, filters=64, pooling=True)
x = residual_block(x, filters=128, pooling=False)

x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

### Batch normalization

### Depthwise separable convolutions

### Putting it together: A mini Xception-like model

In [7]:
# import kagglehub

# kagglehub.login()

In [8]:
# import zipfile

# download_path = kagglehub.competition_download("dogs-vs-cats")

# with zipfile.ZipFile(download_path + "/train.zip", "r") as zip_ref:
#     zip_ref.extractall(".")

UnauthenticatedError: User is not authenticated

In [9]:
from datasets import load_dataset
from PIL import Image
import os, pathlib
import numpy as np

# 1) HF dataset
ds = load_dataset("cats_vs_dogs")  # genelde {"train":..., "test":...}

# 2) Label -> klasör adı
# cats_vs_dogs'ta label genelde 0=cat, 1=dog
label_to_name = {0: "cat", 1: "dog"}

# 3) Notebook'un beklediği hedef klasör
new_base_dir = pathlib.Path("dogs_vs_cats_small")

# Temiz kurulum (varsa klasörü silmek istersen)
# import shutil
# if new_base_dir.exists():
#     shutil.rmtree(new_base_dir)

def ensure_dirs():
    for split in ["train", "validation", "test"]:
        for cls in ["cat", "dog"]:
            (new_base_dir / split / cls).mkdir(parents=True, exist_ok=True)

ensure_dirs()

# 4) HF'de tek bir "train" split varsa: train/val/test'e bölelim
# Not: Orijinal notebook 1000 train, 500 val, 1000 test kullanıyor (küçük dataset).
# Burada da benzer sayıları alacağız.
base = ds["train"]
n = len(base)

# karıştır
rng = np.random.default_rng(42)
indices = rng.permutation(n)

n_train = 1000
n_val = 500
n_test = 1000

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train+n_val]
test_idx  = indices[n_train+n_val:n_train+n_val+n_test]

def export_split(split_name, split_indices, start_counter=0):
    counter = start_counter
    for idx in split_indices:
        ex = base[int(idx)]
        img = ex["image"]  # genelde PIL.Image
        label = int(ex["labels"]) if "labels" in ex else int(ex["label"])
        cls = label_to_name[label]

        # notebook benzeri isimlendirme: cat.0.jpg / dog.0.jpg
        fname = f"{cls}.{counter}.jpg"
        out_path = new_base_dir / split_name / cls / fname

        # güvenli RGB kaydet
        if isinstance(img, Image.Image):
            img.convert("RGB").save(out_path, format="JPEG", quality=95)
        else:
            # bazı varyantlarda img dict/array gelebilir
            Image.fromarray(np.array(img)).convert("RGB").save(out_path, format="JPEG", quality=95)

        counter += 1
    return counter

c = 0
c = export_split("train", train_idx, start_counter=c)
c = export_split("validation", val_idx, start_counter=c)
c = export_split("test", test_idx, start_counter=c)

print("Done. Base dir:", new_base_dir.resolve())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/330M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/391M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/23410 [00:00<?, ? examples/s]

Done. Base dir: /content/dogs_vs_cats_small


In [11]:
import os, shutil, pathlib
from keras.utils import image_dataset_from_directory

original_dir = pathlib.Path("train")
new_base_dir = pathlib.Path("dogs_vs_cats_small")

# def make_subset(subset_name, start_index, end_index):
#     for category in ("cat", "dog"):
#         dir = new_base_dir / subset_name / category
#         os.makedirs(dir)
#         fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
#         for fname in fnames:
#             shutil.copyfile(src=original_dir / fname, dst=dir / fname)

# make_subset("train", start_index=0, end_index=1000)
# make_subset("validation", start_index=1000, end_index=1500)
# make_subset("test", start_index=1500, end_index=2500)

batch_size = 64
image_size = (180, 180)
train_dataset = image_dataset_from_directory(
    new_base_dir / "train",
    image_size=image_size,
    batch_size=batch_size,
)
validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation",
    image_size=image_size,
    batch_size=batch_size,
)
test_dataset = image_dataset_from_directory(
    new_base_dir / "test",
    image_size=image_size,
    batch_size=batch_size,
)

Found 1000 files belonging to 2 classes.
Found 500 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.


In [12]:
import tensorflow as tf
from keras import layers

data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
]

def data_augmentation(images, targets):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images, targets

augmented_train_dataset = train_dataset.map(
    data_augmentation, num_parallel_calls=8
)
augmented_train_dataset = augmented_train_dataset.prefetch(tf.data.AUTOTUNE)

In [13]:
import keras

inputs = keras.Input(shape=(180, 180, 3))
x = layers.Rescaling(1.0 / 255)(inputs)
x = layers.Conv2D(filters=32, kernel_size=5, use_bias=False)(x)

for size in [32, 64, 128, 256, 512]:
    residual = x

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    residual = layers.Conv2D(
        size, 1, strides=2, padding="same", use_bias=False
    )(residual)
    x = layers.add([x, residual])

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

In [14]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
history = model.fit(
    augmented_train_dataset,
    epochs=100,
    validation_data=validation_dataset,
)

Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 91s 3s/step - accuracy: 0.5550 - loss: 0.6878 - val_accuracy: 0.5020 - val_loss: 0.7032
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 390ms/step - accuracy: 0.5870 - loss: 0.6631 - val_accuracy: 0.5020 - val_loss: 0.6985
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 10s 369ms/step - accuracy: 0.5910 - loss: 0.6579 - val_accuracy: 0.5020 - val_loss: 0.7017
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 479ms/step - accuracy: 0.6060 - loss: 0.6556 - val_accuracy: 0.5020 - val_loss: 0.6948
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 383ms/step - accuracy: 0.6210 - loss: 0.6410 - val_accuracy: 0.4960 - val_loss: 0.6959
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - accuracy: 0.6490 - loss: 0.6240 - val_accuracy: 0.5020 - val_loss: 0.6937
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 9s 462ms/step - accuracy: 0.6460 - loss: 0.6183 - val_accuracy: 0.5020 - val_loss: 0.6944
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 469ms/step - accuracy: 0.6630 - loss: 0.6108 - val_accur

### Beyond convolution: Vision Transformers